# DocSight RAG — Fine-Tuning Notebook
**Run this on Google Colab Pro (A100 40GB)**

Steps:
1. Mount Google Drive
2. Install dependencies
3. Upload training data to Drive
4. Run fine-tuning for router / finance / medical
5. Convert to GGUF + upload to HuggingFace Hub

In [ ]:
# Mount Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q unsloth
!pip install -q trl datasets peft bitsandbytes accelerate huggingface_hub

In [ ]:
# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Fine-tune Router (Phi-4-mini — ~30 min)

In [ ]:
import json
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

HF_TOKEN = 'YOUR_HF_TOKEN'  # replace or use Colab secrets

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='microsoft/phi-4-mini-instruct',
    max_seq_length=512,
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', use_gradient_checkpointing='unsloth',
)

# Load training data from Drive
with open('/content/drive/MyDrive/docsight/training/data/router_train.json') as f:
    data = json.load(f)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=Dataset.from_list(data),
    args=SFTConfig(
        output_dir='/content/router_checkpoints',
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        dataset_text_field='text',
        max_seq_length=512,
    ),
)
trainer.train()

In [ ]:
# Save router adapter + merged model
model.save_pretrained('/content/router_adapter')
tokenizer.save_pretrained('/content/router_adapter')
model.save_pretrained_merged('/content/router_merged', tokenizer, save_method='merged_16bit')
print('Router saved.')

## 2. Fine-tune Finance Expert (Llama-3.1-8B — ~2-3h)

In [ ]:
del model, tokenizer  # free VRAM
import gc; gc.collect()
import torch; torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='meta-llama/Llama-3.1-8B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', use_gradient_checkpointing='unsloth',
)

with open('/content/drive/MyDrive/docsight/training/data/finance_sft.json') as f:
    data = json.load(f)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=Dataset.from_list(data),
    args=SFTConfig(
        output_dir='/content/finance_checkpoints',
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=25,
        lr_scheduler_type='cosine',
        warmup_steps=50,
        dataset_text_field='text',
        max_seq_length=2048,
    ),
)
trainer.train()

In [ ]:
model.save_pretrained_merged('/content/finance_merged', tokenizer, save_method='merged_16bit')
# Save GGUF for Ollama
model.save_pretrained_gguf('/content/finance_gguf', tokenizer, quantization_method='q4_k_m')
print('Finance expert saved.')

## 3. Fine-tune Medical Expert (BioMistral-7B — ~2-3h)

In [ ]:
del model, tokenizer
import gc; gc.collect()
import torch; torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='BioMistral/BioMistral-7B',
    max_seq_length=2048,
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', use_gradient_checkpointing='unsloth',
)

with open('/content/drive/MyDrive/docsight/training/data/medical_sft.json') as f:
    data = json.load(f)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=Dataset.from_list(data),
    args=SFTConfig(
        output_dir='/content/medical_checkpoints',
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=25,
        lr_scheduler_type='cosine',
        warmup_steps=50,
        dataset_text_field='text',
        max_seq_length=2048,
    ),
)
trainer.train()

In [ ]:
model.save_pretrained_gguf('/content/medical_gguf', tokenizer, quantization_method='q4_k_m')
print('Medical expert saved.')

## 4. Upload GGUF to HuggingFace Hub
Then pull with Ollama on your local machine.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

# Upload finance GGUF
api.upload_folder(
    folder_path='/content/finance_gguf',
    repo_id='YOUR_HF_USERNAME/docsight-finance-expert-gguf',
    repo_type='model',
)

# Upload medical GGUF  
api.upload_folder(
    folder_path='/content/medical_gguf',
    repo_id='YOUR_HF_USERNAME/docsight-medical-expert-gguf',
    repo_type='model',
)

print('Models uploaded to HuggingFace Hub!')
print('Now run: ollama pull hf.co/YOUR_HF_USERNAME/docsight-finance-expert-gguf')